## Success criteria (written before running any results)

The models count as having a tradable edge only if ALL of these hold across the
8 walk-forward folds (about March 2023 to November 2026):

1. Pooled accuracy of at least 51.0% (the margin of error on 32,000 predictions
   is about ±0.55 points), and above 50% in at least 6 of the 8 folds.
2. The model beats the simple "opposite of last hour" baseline on average.
3. After 0.1% fees per trade, net return is positive in at least 6 of 8 folds
   and beats holding cash overall.

Otherwise the conclusion is: no tradable edge, same as before.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from app.features import FEATURE_COLS, add_target, make_features

raw = pd.read_csv("data/btc_1h_long.csv", parse_dates=["time"])

# Fill the missing hour with a flat candle, so rolling windows stay in hourly units
raw = raw.set_index("time").asfreq("1h")
missing = raw["close"].isna()
raw["close"] = raw["close"].ffill()
for c in ["open", "high", "low"]:
    raw[c] = raw[c].fillna(raw["close"])
raw["volume"] = raw["volume"].fillna(0)
raw = raw.reset_index()
print("Filled missing candles:", int(missing.sum()))

data = add_target(make_features(raw))
print(data.shape, data["time"].min(), "->", data["time"].max())
print("Share of up hours:", round(data["next_up"].mean() * 100, 1), "%")

Filled missing candles: 1
(43778, 10) 2021-11-10 14:00:00 -> 2026-11-08 15:00:00
Share of up hours: 50.4 %


In [2]:
tscv = TimeSeriesSplit(n_splits=8, test_size=4000)

for i, (tr, te) in enumerate(tscv.split(data), 1):
    print(f"Fold {i}: train {len(tr):>6} rows | "
          f"test {data['time'].iloc[te[0]].date()} -> {data['time'].iloc[te[-1]].date()}")

Fold 1: train  11778 rows | test 2023-03-16 -> 2023-08-29
Fold 2: train  15778 rows | test 2023-08-30 -> 2024-02-12
Fold 3: train  19778 rows | test 2024-02-12 -> 2024-07-28
Fold 4: train  23778 rows | test 2024-07-28 -> 2025-01-10
Fold 5: train  27778 rows | test 2025-01-11 -> 2025-06-26
Fold 6: train  31778 rows | test 2025-06-26 -> 2025-12-10
Fold 7: train  35778 rows | test 2025-12-10 -> 2026-05-25
Fold 8: train  39778 rows | test 2026-05-26 -> 2026-11-08


In [3]:
def acc(pred, actual):
    return round((np.asarray(pred) == np.asarray(actual)).mean() * 100, 1)

rows = []
for i, (tr_idx, te_idx) in enumerate(tscv.split(data), 1):
    tr, te = data.iloc[tr_idx], data.iloc[te_idx]
    Xtr, ytr = tr[FEATURE_COLS], tr["next_up"]
    Xte, yte = te[FEATURE_COLS], te["next_up"]

    lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
    xg = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8, random_state=42).fit(Xtr, ytr)

    rows.append({
        "fold": i,
        "test_start": te["time"].iloc[0].date(),
        "up_share": round(yte.mean() * 100, 1),
        "majority": acc(np.full(len(te), int(ytr.mean() > 0.5)), yte),
        "opposite_last": acc(1 - (te["return"] > 0).astype(int), yte),
        "logreg": acc(lr.predict(Xte), yte),
        "xgboost": acc(xg.predict(Xte), yte),
    })

wf = pd.DataFrame(rows)
print(wf.to_string(index=False))

cols = ["majority", "opposite_last", "logreg", "xgboost"]
print("\nAverage (pooled) accuracy:")
print(wf[cols].mean().round(2).to_string())
print("\nFolds above 50%:")
print((wf[cols] > 50).sum().to_string())

 fold test_start  up_share  majority  opposite_last  logreg  xgboost
    1 2023-03-16      50.4      49.6           54.1    53.0     53.8
    2 2023-08-30      51.6      51.6           54.0    54.1     55.4
    3 2024-02-12      51.4      51.4           53.4    52.6     53.8
    4 2024-07-28      51.1      51.1           53.2    53.8     53.2
    5 2025-01-11      50.2      50.2           51.5    51.4     52.3
    6 2025-06-26      50.4      50.4           51.0    52.2     52.4
    7 2025-12-10      49.9      49.9           51.9    52.0     52.4
    8 2026-05-26      49.8      49.8           52.6    51.8     52.3

Average (pooled) accuracy:
majority         50.50
opposite_last    52.71
logreg           52.61
xgboost          53.20

Folds above 50%:
majority         5
opposite_last    8
logreg           8
xgboost          8


In [4]:
FEE = 0.001   # 0.1% per trade
THRESHOLDS = [0.50, 0.55, 0.60]


def fee_backtest(prob, te, threshold, fee=FEE):
    """Hold BTC for the next hour whenever P(up) > threshold; pay a fee on every switch."""
    pos = pd.Series((prob > threshold).astype(int), index=te.index)
    trades = (pos - pos.shift(1).fillna(0)).abs()
    net = pos * te["next_ret"] - trades * fee
    return (1 + net).prod() - 1, int(trades.sum())


net, trade_counts = {}, {}
for i, (tr_idx, te_idx) in enumerate(tscv.split(data), 1):
    tr, te = data.iloc[tr_idx], data.iloc[te_idx]
    Xtr, ytr = tr[FEATURE_COLS], tr["next_up"]
    Xte = te[FEATURE_COLS]

    lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
    xg = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8, random_state=42).fit(Xtr, ytr)
    probs = {"LogReg": lr.predict_proba(Xte)[:, 1], "XGBoost": xg.predict_proba(Xte)[:, 1]}

    for name, p in probs.items():
        for thr in THRESHOLDS:
            key = f"{name} > {thr:.2f}"
            r, n_trades = fee_backtest(p, te, thr)
            net.setdefault(key, []).append(r * 100)
            trade_counts.setdefault(key, []).append(n_trades)

    net.setdefault("Buy and hold", []).append(((1 + te["next_ret"]).prod() - 1) * 100)
    net.setdefault("Cash", []).append(0.0)

res = pd.DataFrame(net).T
res.columns = [f"F{i}" for i in range(1, 9)]
fold_cols = list(res.columns)
res["folds_up"] = (res[fold_cols] > 0).sum(axis=1)
res["total_%"] = ((1 + res[fold_cols] / 100).prod(axis=1) - 1) * 100
res["avg_trades"] = pd.Series({k: np.mean(v) for k, v in trade_counts.items()})
print(res.round(1).to_string())

                  F1    F2    F3    F4    F5    F6    F7    F8  folds_up  total_%  avg_trades
LogReg > 0.50  -59.2 -57.4 -75.7 -57.2 -71.5 -74.6 -73.8 -70.9         0   -100.0      1240.8
LogReg > 0.55  -24.6 -17.8 -34.2 -24.3 -31.9 -30.0 -44.7 -20.0         0    -93.5       440.8
LogReg > 0.60    2.7   0.9 -16.0  -7.3  -5.1  -3.9 -13.0  -2.3         2    -37.4        67.5
XGBoost > 0.50 -79.2 -62.2 -71.1 -65.6 -76.9 -77.2 -77.5 -77.3         0   -100.0      1494.0
XGBoost > 0.55 -65.3 -56.8 -66.5 -54.5 -68.4 -70.4 -68.3 -61.2         0   -100.0      1139.0
XGBoost > 0.60 -38.3 -35.1 -39.7 -28.9 -35.0 -30.9 -26.5 -14.0         0    -95.1       383.6
Buy and hold    12.4  79.5  35.9  40.4  13.3 -13.6 -17.1  -1.3         5    208.3         NaN
Cash             0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0         0      0.0         NaN
